# Plateau’s Problem for Catenoid

This notebook provides an implementation of the Plateau's problem, which finds a minimal surface shape that connects a set of interfaces.
<!-- More details on this example, can be found in [our paper](https://arxiv.org/abs/2402.14009), Sections 4.1 and A.2. -->

### Imports and setup

In [9]:
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch import optim
from tqdm.notebook import trange
import k3d
import sys
import os
import time
import copy
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from models.model_architecture import GeneralNet
from util.surface_sampling import sample_model_surface_binsearch, sample_model_surface_newton 
from util.error_metrics import chamfer_div, compute_distance 
from util.visualization.utils_mesh import get_mesh
from training.residuals import bind_model, r_data, r_eikonal, r_mean_curvature
from training.modular.residuals import ResidualLibrary, ResidualTerm, compute_loss
from training.modular.optimizers import GaussNewton

torch.manual_seed(0)


device = 'cuda' #if torch.cuda.is_available() else 'cpu'
torch.set_default_device(device)

### True Surface

In [4]:
# Parametric equations for the catenoid in polar coordinates
def catenoid_surface_polar(z, phi, c=1.0):
    x = c * torch.cosh(z / c) * torch.cos(phi)
    y = c * torch.cosh(z / c) * torch.sin(phi)
    return x, y, z

# Level set function for the catenoid
def catenoid_level_set(p, c=1.0):
    x = p[:, 0]
    y = p[:, 1]
    z = p[:, 2]
    return x**2 + y**2 - (c**2 * torch.cosh(z / c)**2)

def sample_true_surface(n_samples, c=1.0):
    z = torch.linspace(-z_max, z_max, n_samples, dtype=torch.float64)
    phi = torch.linspace(-torch.pi, torch.pi, 2*n_samples, dtype=torch.float64)    
    z, phi = torch.meshgrid(z, phi, indexing='ij')
    x, y, z = catenoid_surface_polar(z.flatten(), phi.flatten(), c)
    points_on_surface = torch.vstack([x, y, z]).T
    return points_on_surface

# Bounds and number of samples
n = 1000
phi = torch.linspace(-torch.pi, torch.pi, n, dtype=torch.float64)
z_max = 1.0
c = 1.0

# Generate boundary points for the upper and lower circles
z_constant_upper = torch.full_like(phi, z_max, dtype=torch.float64)
z_constant_lower = torch.full_like(phi, -z_max, dtype=torch.float64)
x_upper, y_upper, z_upper = catenoid_surface_polar(z_constant_upper, phi, c)
x_lower, y_lower, z_lower = catenoid_surface_polar(z_constant_lower, phi, c)
pts_upper = torch.vstack([x_upper, y_upper, z_upper]).T
pts_lower = torch.vstack([x_lower, y_lower, z_lower]).T
pts_boundary = torch.cat([pts_upper, pts_lower], dim=0)
pts_surface_true = sample_true_surface(64)


# Generate the mesh using the level set function
verts, faces = get_mesh(
    lambda x: catenoid_level_set(x, c),
    N=128, 
    device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

print("True surface:")
fig.display()

True surface:


/home/arturs/anaconda3/envs/GINN/lib/python3.12/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

### Pretraining

In [5]:
model = GeneralNet(ks=[3, 32, 32, 1])
model = model.double()
bind_model(model)
# Generate random training points
num_pretrain_samples = 10000
bounds = torch.tensor([[-1,1], [-1,1], [-1,1]], dtype=torch.float64)
pts_pretrain = torch.rand(num_pretrain_samples, 3, dtype=torch.float64) * (bounds[:,1] - bounds[:,0]) + bounds[:,0]


def pretrain_loss(model, params, pts):
    inputs = pts.to(dtype=torch.float64)
    x, y = pts[:, 0], pts[:, 1]
    targets = x**2 + y**2 - 1
    preds = model(inputs).squeeze(1)
    return 0.5 * (preds - targets).square().mean()

# Pretraining loop
pretrain_optimizer = optim.Adam(model.parameters(), lr=1e-3)
num_pretrain_iters = 1000

for i in range(num_pretrain_iters):
    pretrain_optimizer.zero_grad()
    loss = pretrain_loss(model, model.params, pts_pretrain)
    loss.backward()
    pretrain_optimizer.step()
    
    if i % 100 == 0:
        print(f"Pretrain Iter {i}: Loss= {loss.item():.6f}")

print("Pretraining completed!")

verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-3, -3, -1.5], dtype=torch.float64),
    bbox_max=torch.tensor([3, 3, 1.5], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)
fig.display()


Pretrain Iter 0: Loss= 0.091712
Pretrain Iter 100: Loss= 0.047754
Pretrain Iter 200: Loss= 0.002024
Pretrain Iter 300: Loss= 0.000846
Pretrain Iter 400: Loss= 0.000410
Pretrain Iter 500: Loss= 0.000210
Pretrain Iter 600: Loss= 0.000118
Pretrain Iter 700: Loss= 0.000073
Pretrain Iter 800: Loss= 0.000058
Pretrain Iter 900: Loss= 0.000039
Pretraining completed!


/home/arturs/anaconda3/envs/GINN/lib/python3.12/site-packages/traittypes/traittypes.py:97: UserWarning: Given trait value dtype "int32" does not match required type "uint32". A coerced copy has been created.
  warnings.warn(


Output()

### Gauss-Newton modular implementation

Key idea:
- `ResidualLibrary` implements the different residuals, e.g., data, eikonal ...
- `ResidualTerm` implements standard transformations to apply the residuals, e.g., vectorization, Jacobian ...
- `res_term` dictionary registers the residuals that define the specific optimization problem
- `GaussNewton` and its subroutines become simpler since all residuals share the same abstraction

In [6]:
res_lib = ResidualLibrary(model)

## Example
# x = pts_boundary[0].float()
# res_lib._data(model.params, x, 0)
# res_lib._eikonal(model.params, x)
# res_lib._normal(model.params, x, torch.tensor([.0, .0, .0]))
# res_lib._laplacian(model.params, x)
# res_lib._strain(model.params, x)

In [7]:
pts_eikonal = 2 * torch.rand([1000, 3], dtype=torch.float64) - 1
pts_eikonal = torch.cat((pts_eikonal, pts_boundary))

## Select the residual terms defining the specific problem
## (incl. weight, points, and (optional) target values)
res_terms = {
    "data":
        ResidualTerm(res_lib._data,           1.0,  pts_boundary, vals=0),
    "eikonal":
        ResidualTerm(res_lib._eikonal,        1e-3, pts_eikonal,        ),
    "mean_curvature":
        ResidualTerm(res_lib._mean_curvature, 1.0,  pts_boundary,       ),
}


## Example
# model = model.double()
# params = model.params

# loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
# print(loss.item())
# for key, l in unweighted_losses.items():
#     print(key, l.item())
    
##
# res_terms["data"].grad_theta_r(params)["fcs.0.weight"].shape
# res_terms["data"].weighted_loss(params)

In [ ]:
## Example
# for key in res_terms:
#     print(compute_JTJ_per_residual(params, res_terms[key]).shape)
# compute_JTJ(params, res_terms)

### Main training loop of the modular GN

In [10]:
model = model.double()
params = model.params
loss_over_time = {}
distance_over_time = {}
chamfer_over_time = {}
best_loss = float('inf')

optim = GaussNewton(model, res_terms, lr=1e-1, regularization=1e-6, do_line_search=False)
start_time = time.time()
current_time = 0

for i in (pbar:=trange(100000)):
    if current_time > 1200:
        break
    optim.zero_grad()


    ## Update weights
    if i == 500:
        loss_weights = {"data": 1.0, "eikonal": 0.0, "mean_curvature": 1.0}
        for key in res_terms:
            res_terms[key].weight = loss_weights[key]

    ## Update surface points
    pts_surface = sample_model_surface_binsearch(model, pts_boundary, bound_limit=2)
    res_terms["mean_curvature"].points = pts_surface
    
    ## Evaluate the losses
    loss, unweighted_losses = compute_loss(params, res_terms, return_unweighted_losses=True)
        
    loss.backward()
    
    with torch.no_grad():
        loss_metric = unweighted_losses["data"] + unweighted_losses["mean_curvature"]

        current_time = time.time() - start_time
        loss_over_time[current_time] = loss_metric.item()
        distance_over_time[current_time] = compute_distance(model.double(), catenoid_level_set, pts_surface, pts_surface_true, 1.0)
        chamfer_over_time[current_time] = chamfer_div(model, pts_surface_true)

        if loss_metric.item() < best_loss:
            best_loss = loss_metric.item()
            best_model_state = copy.deepcopy(model.state_dict())

        unweighted_losses_str = " ".join(f"{key}: {l.item():.2e}" for key, l in unweighted_losses.items())
        pbar.set_description(unweighted_losses_str + " "
                            f"error: {chamfer_over_time[current_time]:.2e} "
                            f"nof_pts: {len(pts_surface)}"
                            )
    optim.step()

# Optional: Load the best model after training
# if best_model_state is not None:
#     model.load_state_dict(best_model_state)
#     print(f"Best model loaded with loss {best_loss}")

plt.plot(loss_over_time.keys(), loss_over_time.values())
plt.semilogy()
plt.show()

  0%|          | 0/100000 [00:00<?, ?it/s]

KeyboardInterrupt: 

### Visualize the result

In [ ]:
verts, faces = get_mesh(
    model.float(), N=128, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)
fig += k3d.mesh(verts, faces, color=0xbbbbbb, side='double', flat_shading=False)
fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)
fig += k3d.points(pts_surface_true.cpu().detach(), point_size=0.05)

fig.display()

In [ ]:
verts, faces = get_mesh(
    model.float(), N=256, device=device,
    bbox_min=torch.tensor([-2, -2, -1.25], dtype=torch.float64),
    bbox_max=torch.tensor([2, 2, 1.25], dtype=torch.float64),
    chunks=2
)

model.double()
mean_curvatures = r_mean_curvature(params, torch.tensor(verts, dtype=torch.float64)).squeeze(1).abs()

fig = k3d.plot(height=800, grid_visible=False, camera_fov=1.0)

color_map = k3d.basic_color_maps.Jet
color_range = [0, 0.005]

fig += k3d.mesh(
    verts, faces, 
    attribute=mean_curvatures.cpu().detach().numpy().astype(np.float32),
    color_range=color_range,
    color_map=color_map,
    side='double',
    flat_shading=False
)

fig += k3d.points(pts_boundary.cpu().detach(), color=0x00ff00, point_size=0.05)

fig.display()